In [1]:
import glob
import torch
import os
import json
from pathlib import Path
from PIL import Image
import random
from datasets import Dataset
from transformers import DonutProcessor, VisionEncoderDecoderModel
from transformers import VisionEncoderDecoderConfig
from transformers import GenerationConfig

from typing import Any, List, Tuple
from datasets.info import DatasetInfo
from datasets.splits import NamedSplit
from datasets.table import Table


import lightning as L
from torch.utils.data import DataLoader

from pathlib import Path
import re
from nltk import edit_distance
import numpy as np
import math

from torch.nn.utils.rnn import pad_sequence
from torch.optim.lr_scheduler import LambdaLR


import mlflow

In [2]:
image_height=560
image_width=560


# the donut model uses CrossEntropyLoss as loss function for training
# we need to give the ignore-index for pad_token, let CrossEntropyLoss function ignore the loss values of this part
loss_ignore_index=-100

In [3]:
# !pip install -U transformers

In [4]:
# enable GPU
if torch.backends.mps.is_available():
    print("✅ MPS (Metal Performance Shaders) is available!")
    device = torch.device("mps") 
else:
    print("MPS is not available. Using CPU.")
    device = torch.device("cpu")

print(f"PyTorch version: {torch.__version__}")

x = torch.rand(3, 3).to(device)
print(f"Tensor on device: {x.device}")

✅ MPS (Metal Performance Shaders) is available!
PyTorch version: 2.2.0
Tensor on device: mps:0


In [5]:
config = VisionEncoderDecoderConfig.from_pretrained("naver-clova-ix/donut-base")
processor = DonutProcessor.from_pretrained("naver-clova-ix/donut-base")


Could not find image processor class in the image processor config or the model config. Loading based on pattern matching with the model's feature extractor configuration. Please open a PR/issue to update `preprocessor_config.json` to use `image_processor_type` instead of `feature_extractor_type`. This warning will be removed in v4.40.


In [6]:
processor.tokenizer

XLMRobertaTokenizerFast(name_or_path='naver-clova-ix/donut-base', vocab_size=57522, model_max_length=1000000000000000019884624838656, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'sep_token': '</s>', 'pad_token': '<pad>', 'cls_token': '<s>', 'mask_token': '<mask>', 'additional_special_tokens': ['<s_iitcdip>', '<s_synthdog>']}, clean_up_tokenization_spaces=True),  added_tokens_decoder={
	0: AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	57521: AddedToken("<mask>", rstrip=False, lstrip=True, single_word=False, normalized=True, special=Tru

In [7]:
annotations_dir="/Users/yiding/personal_projects/ML/github_repo/donut/data/image_classes/vertical_bar_annotations/"
annotations_files=glob.glob(annotations_dir+"*")
print(annotations_files[0])
print(len(annotations_files))

image_dir="/Users/yiding/personal_projects/ML/github_repo/donut/data/image_classes/vertical_bar/"
images_files=glob.glob(image_dir+"*")
print(images_files[0])
print(len(images_files))

/Users/yiding/personal_projects/ML/github_repo/donut/data/image_classes/vertical_bar_annotations/75c0449f6917.json
19189
/Users/yiding/personal_projects/ML/github_repo/donut/data/image_classes/vertical_bar/b2ab3b743d4e.jpg
19189


In [8]:
# # load data in s3 into sagemaker domain
# !aws s3 cp s3://birdclef/donut/ ./donut/ --recursive

In [ ]:
# format data with Huggingface dataset

annotations_dir = Path("./donut/vertical_bar_annotations")
images_dir = Path("./donut/vertical_bar")
annotations_files = annotations_dir.glob("*.json")

ds = Dataset.from_dict({"annotation_path": list(map(str, list(annotations_files)))})

#---------------------

def parse_image(batch):
    """
    Processes a batch of data by loading images corresponding to annotation file paths.

    This function takes a batch containing annotation paths, extracts the file IDs from these paths,
    and uses these IDs to load the corresponding image files. Each image is opened and stored in a list.
    The function assumes that the image files are named with the same file ID as the annotation files
    and are located in a directory specified by `images_dir`.

    Parameters:
    batch (dict): A dictionary containing a key "annotation_path", which maps to a list of paths to annotation files.
                  Each path is expected to be in string format.

    Returns:
    dict: A dictionary with a key "images", which maps to a list of loaded images corresponding to the annotation paths.

    Note:
    - The variable `images_dir` should be defined outside this function and must point to the directory containing the image files.
    - The function does not close the images; users of the function should handle this as necessary.
    - This function assumes that the image files have a '.jpg' extension.
    """
    annotation_paths = batch["annotation_path"]

    images = []

    # Traverse all JSON file paths
    for path in annotation_paths:
        # Extract the file ID from the annotation path
        file_id = Path(path).stem

        # Load the corresponding image
        image = Image.open(str(images_dir / f"{file_id}.jpg"))
        images.append(image)

    return {"images": images}


parse_image({"annotation_path": ["./donut/vertical_bar_annotations/00cc0db43de7.json"]})

# add images into dataset
# map corresponding image to annotation file
ds1 = ds.map(parse_image, batched=True, num_proc=8)


# release memory
del ds

#--------------------

def parse_jsonl(batch):
    """
    convert json file into string with special tokens.
    each filed should be wrapped up by special tokens
    """
    annotation_paths = batch['annotation_path']
    gt_strings = []

    for path in annotation_paths:
        with open(path, 'r', encoding='utf-8') as f:
            data = json.load(f)

        # 1. grab field content
        title = data.get("title", "null")
        x_axis_title = data.get("x_axis_title", "null")
        y_axis_title = data.get("y_axis_title", "null")
        x_ticks = data.get("x_ticks", [])
        y_ticks = data.get("y_ticks", [])
        data_series = data.get("data_series", [])

        # 2. construct special tokens
        x_tick_str = "|".join(str(x) for x in x_ticks)
        y_tick_str = "|".join(str(y) for y in y_ticks)
        series_str = "|".join(f"{item['x']}:{round(float(item['y']), 2)}" for item in data_series)

        content = (
            f"<title>{title}</title>"
            f"<x_axis_title>{x_axis_title}</x_axis_title>"
            f"<y_axis_title>{y_axis_title}</y_axis_title>"
            f"<x_ticks>{x_tick_str}</x_ticks>"
            f"<y_ticks>{y_tick_str}</y_ticks>"
            f"<data_series>{series_str}</data_series>"
        )

        gt_strings.append(content)

    return {
        "gt_string": gt_strings
    }


# map parse_gt into Dataset
ds2=ds1.map(parse_jsonl, batched=True, num_proc=8)


# Dataset({
 #   features: ['annotation_path', 'images', 'gt_string'],
 #   num_rows: 19189
#})


# release memory
del ds1


# train test split
ds3 = ds2.train_test_split(test_size=0.3, shuffle=True, seed=42)

# DatasetDict({
#    train: Dataset({
#        features: ['annotation_path', 'images', 'gt_string'],
#        num_rows: 13432
#    })
#    test: Dataset({
#        features: ['annotation_path', 'images', 'gt_string'],
#        num_rows: 5757
#    })
#})



# release memory
del ds2

In [ ]:
# check the label string len, define the model's decoder token length.

lengths = [len(processor.tokenizer(gt)["input_ids"]) for gt in ds2["gt_string"]]

import numpy as np
print("📊 Max length:", np.max(lengths))
print("📊 95th percentile:", np.percentile(lengths, 95))
print("📊 Mean:", np.mean(lengths))


max_length=int(np.percentile(lengths, 95)) + 20
# max_length=286

In [9]:
## some extra special tokens
title_start_token="<title>"
title_end_token="</title>"
x_axis_title_start_token="<x_axis_title>"
x_axis_title_end_token="</x_axis_title>"
y_axis_title_start_token="<y_axis_title>"
y_axis_title_end_token="</y_axis_title>"
x_ticks_start_token="<x_ticks>"
x_ticks_end_token="</x_ticks>"
y_ticks_start_token="<y_ticks>"
y_ticks_end_token="</y_ticks>"
data_series_start_token="<data_series>"
data_series_end_token="</data_series>"


special_tokens = [
    "<title>", "</title>",
    "<x_axis_title>", "</x_axis_title>",
    "<y_axis_title>", "</y_axis_title>",
    "<x_ticks>", "</x_ticks>",
    "<y_ticks>", "</y_ticks>",
    "<data_series>", "</data_series>"
]

In [11]:
# setup the config file
config.encoder.image_size = (image_height, image_width)
# config.decoder.max_length = max_length
config.decoder.max_length = 286
# add necessary tokens into config
config.decoder_start_token_id = processor.tokenizer.convert_tokens_to_ids("<s>")
config.pad_token_id = processor.tokenizer.convert_tokens_to_ids("<pad>")
config.eos_token_id = processor.tokenizer.convert_tokens_to_ids("</s>")

In [12]:
# setup the processor

# update image size
processor.image_processor.size = {
    "height": image_height,
    "width": image_width,
}

# add special tokens into processor
processor.tokenizer.add_special_tokens({"additional_special_tokens": special_tokens})

12

In [13]:
from torch.utils.data import Dataset
class DonutDataset(Dataset):
    def __init__(self, 
                 dataset,
                 split,
                 max_length,
                 loss_ignore_index,
    ):
        super().__init__()

        self.max_length = max_length
        self.split = split
        self.dataset = dataset[self.split]
        self.loss_ignore_index=loss_ignore_index
        self.dataset_length = len(self.dataset)

        self.gt_token_sequences=[]

        for sample in self.dataset:
            self.gt_token_sequences.append([sample['gt_string']])
        

    def __len__(self):
        return self.dataset_length

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:

        sample = self.dataset[idx]

        # inputs
        pixel_values = processor(
            sample["images"], random_padding=self.split == "train", return_tensors="pt"
        ).pixel_values
        pixel_values = pixel_values.squeeze()

        # targets
        # print(self.gt_token_sequences)
        # print(len(self.gt_token_sequences))
        target_sequence = random.choice(self.gt_token_sequences[idx])
        input_ids = processor.tokenizer(
            target_sequence,
            add_special_tokens=False,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )["input_ids"].squeeze(0)

        labels = input_ids.clone()
        labels[labels == processor.tokenizer.pad_token_id] = (
            self.loss_ignore_index
        )

        # print("input_ids:", input_ids.tolist())
        # print("labels   :", labels.tolist())

        return pixel_values, labels, target_sequence

In [14]:
# train_dataset = DonutDataset(
#     dataset=ds3,
#     max_length=max_length,
#     split="train",
#     loss_ignore_index=loss_ignore_index
# )
    

# val_dataset = DonutDataset(
#     dataset=ds3,
#     max_length=max_length,
#     split="test",
#     loss_ignore_index=loss_ignore_index
# )

In [15]:
class DonutDataModule(L.LightningDataModule):
    def __init__(
        self,
        dataset,
        processor,
        max_length=768,
        loss_ignore_index=-100,
        train_batch_size=32,
        val_batch_size=32,
        num_workers=0,
    ):
        super().__init__()
        self.dataset = dataset
        self.processor = processor
        self.max_length = max_length
        self.loss_ignore_index = loss_ignore_index
        self.train_batch_size = train_batch_size
        self.val_batch_size = val_batch_size
        self.num_workers = num_workers

    def setup(self, stage=None):
        self.train_dataset = DonutDataset(
            dataset=self.dataset,
            split="train",
            max_length=self.max_length,
            loss_ignore_index=self.loss_ignore_index,
        )
        self.val_dataset = DonutDataset(
            dataset=self.dataset,
            split="test",
            max_length=self.max_length,
            loss_ignore_index=self.loss_ignore_index,
        )

    def train_dataloader(self):
        return DataLoader(
            self.train_dataset,
            batch_size=self.train_batch_size,
            shuffle=True,
            num_workers=self.num_workers,
        )

    def val_dataloader(self):
        return DataLoader(
            self.val_dataset,
            batch_size=self.val_batch_size,
            shuffle=False,
            num_workers=self.num_workers,
        )

In [16]:
model = VisionEncoderDecoderModel.from_pretrained("naver-clova-ix/donut-base", config=config)


In [18]:
# extend decoder embedding layer
model.decoder.resize_token_embeddings(len(processor.tokenizer))

model.config.eos_token_id = processor.tokenizer.eos_token_id
model.config.decoder_start_token_id = processor.tokenizer.convert_tokens_to_ids("<s>")
# model.config.max_length =max_length
model.config.max_length =286
model.config.pad_token_id=processor.tokenizer.pad_token_id

model.config.encoder.pad_token_id=processor.tokenizer.pad_token_id
model.config.decoder.bos_token_id=processor.tokenizer.bos_token_id
model.config.encoder.bos_token_id=processor.tokenizer.bos_token_id
model.config.update({"special_tokens": processor.tokenizer.additional_special_tokens})

In [19]:
print("decoder_start_token_id:", model.config.decoder_start_token_id)
print("token:", processor.tokenizer.convert_ids_to_tokens(model.config.decoder_start_token_id))
print("model.config.max_length:", model.config.max_length)
print("vocab size:", len(processor.tokenizer))
print(model.config.pad_token_id)
print(processor.tokenizer.convert_tokens_to_ids("<title>"))
print(model.decoder.get_input_embeddings().weight.shape)  

decoder_start_token_id: 0
token: <s>
model.config.max_length: 286
vocab size: 57537
1
57525
torch.Size([57537, 1024])


In [20]:
from torch.optim.lr_scheduler import LambdaLR
import math

def get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps, num_cycles=0.5, last_epoch=-1):
    def lr_lambda(current_step):
        if current_step < warmup_steps:
            return float(current_step) / float(max(1, warmup_steps))
        progress = float(current_step - warmup_steps) / float(max(1, total_steps - warmup_steps))
        return max(0.0, 0.5 * (1.0 + math.cos(math.pi * num_cycles * 2.0 * progress)))
    return LambdaLR(optimizer, lr_lambda, last_epoch=last_epoch)

In [21]:
class DonutModelPLModule(L.LightningModule):
    def __init__(self, config, processor, model):
        super().__init__()
        self.config = config
        self.processor = processor
        self.model = model

        if self.config.get("freeze_encoder", True):
            self.freeze_encoder()

        self.print_trainable_parameters()

        self.generation_cfg = GenerationConfig(
            max_length=self.config.get("max_length", 768),
            early_stopping=True,
            do_sample=False,
            num_beams=4,
            use_cache=True,
            bad_words_ids=[[self.processor.tokenizer.unk_token_id]],
            pad_token_id=self.processor.tokenizer.pad_token_id,
            eos_token_id=self.processor.tokenizer.eos_token_id,
            return_dict_in_generate=True,
        )

    def freeze_encoder(self):
        for param in self.model.encoder.parameters():
            param.requires_grad = False

    def print_trainable_parameters(self):
        trainable_params = sum(p.numel() for p in self.parameters() if p.requires_grad)
        total_params = sum(p.numel() for p in self.parameters())
        print(f"Trainable parameters: {trainable_params} / {total_params}")

    def training_step(self, batch, batch_idx):
        pixel_values, labels, _ = batch
        pixel_values = pixel_values.to(self.device)
        labels = labels.to(self.device)

        outputs = self.model(pixel_values, labels=labels)
        loss = outputs.loss

        # mlflow.log_metric("Training Batch Loss", loss.item(), step=batch_idx)
        self.log("train_loss", loss)
        return loss

    def validation_step(self, batch, batch_idx, dataset_idx=0):
        pixel_values, labels, answers = batch
        pixel_values = pixel_values.to(self.device)
        labels = labels.to(self.device)
    
        # CrossEntropyLoss (token-level)
        with torch.no_grad():
            outputs = self.model(pixel_values, labels=labels)
            val_loss = outputs.loss
    
            # Log to MLflow & Lightning both
            # mlflow.log_metric("val_loss", val_loss.item(), step=batch_idx)
            self.log("val_loss", val_loss)
    
        # Optional: Edit Distance evaluation
        if self.config.get("log_edit_distance", False):
            decoder_input_ids = torch.full(
                (pixel_values.size(0), 1),
                self.model.config.decoder_start_token_id,
                device=self.device,
            )
    
            generated = self.model.generate(
                pixel_values,
                decoder_input_ids=decoder_input_ids,
                generation_config=self.generation_cfg,
            )
    
            predictions = self.processor.tokenizer.batch_decode(
                generated.sequences, skip_special_tokens=True
            )
    
            scores = []
            for pred, ans in zip(predictions, answers):
                denom = max(len(pred), len(ans))
                score = edit_distance(pred, ans) / denom if denom > 0 else 1.0
                scores.append(score)
    
            # 📝 Log Edit Distance
            avg_score = np.mean(scores)
            # mlflow.log_metric("val_edit_distance", avg_score, step=batch_idx)
            self.log("val_edit_distance", avg_score)
    
        return {"val_loss": val_loss}


    def predict_step(self, batch, batch_idx, dataloader_idx=0):
        pixel_values = batch.to(self.device)
        batch_size = pixel_values.shape[0]

        decoder_input_ids = torch.full(
            (batch_size, 1),
            self.model.config.decoder_start_token_id,
            dtype=torch.long,
            device=self.device,
        )

        outputs = self.model.generate(
            pixel_values,
            decoder_input_ids=decoder_input_ids,
            generation_config=self.generation_cfg,
        )

        predictions = self.processor.tokenizer.batch_decode(
            outputs.sequences, skip_special_tokens=True
        )

        return predictions

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.config.get("lr", 5e-5))
    
        total_steps = self.config.get("max_epochs", 30) * self.config.get("steps_per_epoch", 1000)
        warmup_steps = self.config.get("warmup_steps", int(0.1 * total_steps))
    
        scheduler = get_cosine_schedule_with_warmup(
            optimizer,
            warmup_steps=warmup_steps,
            total_steps=total_steps,
            num_cycles=0.5
        )
    
        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "interval": "step",  # update every step
                "frequency": 1,
            }
        }

In [26]:
train_config = {"max_epochs":30,
          "val_check_interval":1, # how many times we want to validate during an epoch
          "check_val_every_n_epoch":1,
          "gradient_clip_val":0.1,
          "lr":1e-6,
          "train_batch_sizes": [32],
          "val_batch_sizes": [32],
          "num_nodes": 4,
          # "warmup_steps": 300,
          "result_path": "./donut/result",
          "log_edit_distance": False,
          "verbose": True,
          }

In [23]:
model_module = DonutModelPLModule(train_config, processor, model)

Trainable parameters: 127683584 / 201864312


In [46]:
model.config

VisionEncoderDecoderConfig {
  "_name_or_path": "naver-clova-ix/donut-base",
  "architectures": [
    "VisionEncoderDecoderModel"
  ],
  "decoder": {
    "_name_or_path": "",
    "activation_dropout": 0.0,
    "activation_function": "gelu",
    "add_cross_attention": true,
    "add_final_layer_norm": true,
    "architectures": null,
    "attention_dropout": 0.0,
    "bad_words_ids": null,
    "begin_suppress_tokens": null,
    "bos_token_id": 0,
    "chunk_size_feed_forward": 0,
    "classifier_dropout": 0.0,
    "cross_attention_hidden_size": null,
    "d_model": 1024,
    "decoder_attention_heads": 16,
    "decoder_ffn_dim": 4096,
    "decoder_layerdrop": 0.0,
    "decoder_layers": 4,
    "decoder_start_token_id": null,
    "diversity_penalty": 0.0,
    "do_sample": false,
    "dropout": 0.1,
    "early_stopping": false,
    "encoder_attention_heads": 16,
    "encoder_ffn_dim": 4096,
    "encoder_layerdrop": 0.0,
    "encoder_layers": 12,
    "encoder_no_repeat_ngram_size": 0,
    "e

In [ ]:
model.generation_config

In [ ]:
dm = DonutDataModule(
    dataset=ds3,
    processor=processor,
    max_length=max_length,
    loss_ignore_index=loss_ignore_index,
    train_batch_size=32,
    val_batch_size=32,
    num_workers=2
)

In [ ]:
from lightning.pytorch import Trainer
from lightning.pytorch.callbacks import Callback, ModelCheckpoint, EarlyStopping

# create EarlyStopping callback
early_stop_callback = EarlyStopping(monitor="val_loss", patience=6, verbose=False, mode="min")

# create ModelCheckpoint callback
checkpoint_callback = ModelCheckpoint(
    dirpath="./donut/result",  
    filename="{epoch}-{val_loss:.2f}",  
    save_top_k=1,  
    verbose=True,
    monitor="val_loss",  
    mode="min", 
)

# initilize Trainer
trainer = Trainer(
        accelerator="gpu",
        devices=1,
        max_epochs=train_config.get("max_epochs"),
        # val_check_interval=train_config.get("val_check_interval"),
        check_val_every_n_epoch=train_config.get("check_val_every_n_epoch"),
        gradient_clip_val=train_config.get("gradient_clip_val"),
        precision=16,  # mixed precision
        num_sanity_val_steps=0,
        callbacks=[early_stop_callback, checkpoint_callback],  # add callbacks
)

trainer.fit(model_module,datamodule=dm)
# # use mlflow
# with mlflow.start_run():
#     # record parameters
#     mlflow.log_params(config)

#     # train the model
#     trainer.fit(model_module,datamodule=dm)

#     # log model
#     mlflow.pytorch.log_model(model_module.model, "model")
#     mlflow.end_run()